<a href="https://colab.research.google.com/github/AvirupVIP/SIGNIFY---Signature-Forgery-Detection-System/blob/master/MobileNet_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install kaggle

In [5]:
import os
import json

kaggle_config = {
  "username": "sandandipto",
  "key": "KGAT_d5a3d64f806d070d376a3862e1c6f371"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_config, f)

!chmod 600 /root/.kaggle/kaggle.json

In [6]:
!kaggle config view

Configuration values from /root/.kaggle
- username: sandandipto
- path: None
- proxy: None
- competition: None


In [7]:
!kaggle datasets download -d shreelakshmigp/cedardataset

Dataset URL: https://www.kaggle.com/datasets/shreelakshmigp/cedardataset
License(s): unknown
 99% 240M/242M [00:00<00:00, 618MB/s] 
100% 242M/242M [00:00<00:00, 670MB/s]


In [8]:
!kaggle datasets download -d tienen/handwritten-signature-verification


Dataset URL: https://www.kaggle.com/datasets/tienen/handwritten-signature-verification
License(s): CC0-1.0
 99% 2.09G/2.10G [00:21<00:00, 242MB/s]
100% 2.10G/2.10G [00:21<00:00, 106MB/s]


In [9]:
!ls

cedardataset.zip  handwritten-signature-verification.zip  sample_data


In [10]:
import zipfile
import os

os.makedirs("/content/datasets", exist_ok=True)

zip_files = [
    "cedardataset.zip",
    "handwritten-signature-verification.zip"
]

for z in zip_files:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall("/content/datasets")

In [11]:
!ls /content/datasets

assignments_07-02-2022.tsv  data  signatures


In [12]:
combined_path = "/content/combined_signature_dataset"

real_path = os.path.join(combined_path,"real")
forged_path = os.path.join(combined_path,"forged")

os.makedirs(real_path,exist_ok=True)
os.makedirs(forged_path,exist_ok=True)

In [13]:
import shutil

source_root = "/content/datasets"

for root, dirs, files in os.walk(source_root):

    for file in files:

        if file.lower().endswith((".png",".jpg",".jpeg",".bmp",".tif")):

            src = os.path.join(root,file)
            folder_name = root.lower()

            if "forg" in folder_name:
                shutil.copy(src, forged_path)

            elif "real" in folder_name or "genuine" in folder_name:
                shutil.copy(src, real_path)

In [14]:
print("Real Signatures:", len(os.listdir(real_path)))
print("Forged Signatures:", len(os.listdir(forged_path)))

Real Signatures: 3188
Forged Signatures: 4304


In [15]:
import cv2
import numpy as np
from tqdm import tqdm

In [16]:
processed_dir = "/content/preprocessed_signatures"

real_processed = os.path.join(processed_dir,"real")
forged_processed = os.path.join(processed_dir,"forged")

os.makedirs(real_processed,exist_ok=True)
os.makedirs(forged_processed,exist_ok=True)

In [17]:
IMG_SIZE = 224

def preprocess_folder(input_dir, output_dir):

    for img_name in tqdm(os.listdir(input_dir)):

        img_path = os.path.join(input_dir,img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        resized = cv2.resize(gray,(IMG_SIZE,IMG_SIZE))

        normalized = resized / 255.0

        save_path = os.path.join(output_dir,img_name)

        cv2.imwrite(save_path, normalized*255)

In [18]:
print("Processing Real Signatures...")
preprocess_folder(real_path,real_processed)

print("Processing Forged Signatures...")
preprocess_folder(forged_path,forged_processed)

Processing Real Signatures...


100%|██████████| 3188/3188 [00:32<00:00, 98.83it/s] 


Processing Forged Signatures...


100%|██████████| 4304/4304 [00:34<00:00, 126.05it/s]


In [19]:
print("Processed Real:", len(os.listdir(real_processed)))
print("Processed Forged:", len(os.listdir(forged_processed)))

Processed Real: 3188
Processed Forged: 4303


In [20]:
import shutil

shutil.make_archive(
    "signature_dataset_mobilenet",
    'zip',
    processed_dir
)

'/content/signature_dataset_mobilenet.zip'

In [21]:
from google.colab import files
files.download("signature_dataset_mobilenet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!pip install tensorflow opencv-python tqdm
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.callbacks import EarlyStopping

In [23]:
dataset_path = "/content/preprocessed_signatures"

real_path = os.path.join(dataset_path,"real")
forged_path = os.path.join(dataset_path,"forged")

IMG_SIZE = 224
def load_images(path):

    images = []

    for img in tqdm(os.listdir(path)):

        img_path = os.path.join(path,img)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))

        img = img/255.0

        img = np.expand_dims(img,-1)

        images.append(img)

    return np.array(images)

In [24]:
print("Loading Genuine Signatures")
genuine_images = load_images(real_path)

print("Loading Forged Signatures")
forged_images = load_images(forged_path)

print(genuine_images.shape)
print(forged_images.shape)

Loading Genuine Signatures


100%|██████████| 3188/3188 [00:01<00:00, 1902.61it/s]


Loading Forged Signatures


100%|██████████| 4303/4303 [00:06<00:00, 676.50it/s]


(3188, 224, 224, 1)
(4303, 224, 224, 1)


In [25]:
pairs = []
labels = []

min_samples = min(len(genuine_images), len(forged_images))

for i in range(min_samples):

    # Positive pair
    pairs.append([genuine_images[i], genuine_images[(i+1)%min_samples]])
    labels.append(1)

    # Negative pair
    pairs.append([genuine_images[i], forged_images[i]])
    labels.append(0)

pairs = np.array(pairs)
labels = np.array(labels)

print("Total Pairs:",pairs.shape)

Total Pairs: (6376, 2, 224, 224, 1)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    pairs, labels,
    test_size=0.2,
    random_state=42
)

NameError: name 'train_test_split' is not defined

In [ ]:
X_train_1 = X_train[:,0]
X_train_2 = X_train[:,1]

X_test_1 = X_test[:,0]
X_test_2 = X_test[:,1]

In [1]:
def build_embedding():

    input = Input((224,224,1))

    x = Concatenate()([input,input,input])

    base_model = MobileNet(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    for layer in base_model.layers[:-20]:
        layer.trainable = False

    x = base_model(x)

    x = GlobalAveragePooling2D()(x)

    x = Dense(256,activation='relu')(x)

    x = Dropout(0.3)(x)

    x = Dense(128)(x)

    return Model(input,x)

In [3]:
embedding_model = build_embedding()

embedding_model.summary()

NameError: name 'Input' is not defined

In [ ]:
input1 = Input((224,224,1))
input2 = Input((224,224,1))

embedding1 = embedding_model(input1)
embedding2 = embedding_model(input2)

distance = Lambda(
    lambda tensors: tf.abs(tensors[0] - tensors[1])
)([embedding1, embedding2])

x = Dense(64,activation='relu')(distance)

output = Dense(1,activation='sigmoid')(x)

siamese_model = Model([input1,input2],output)

In [ ]:
siamese_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = siamese_model.fit(

    [X_train_1, X_train_2],
    y_train,

    validation_data=(
        [X_test_1, X_test_2],
        y_test
    ),

    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])

plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train","Validation"])

plt.show()

In [ ]:
def preprocess_test(img_path):

    img = cv2.imread(img_path,cv2.IMREAD_GRAYSCALE)

    img = cv2.resize(img,(224,224))

    img = img/255.0

    img = np.expand_dims(img,-1)

    img = np.expand_dims(img,0)

    return img

In [ ]:
def verify_signature(img1,img2):

    img1 = preprocess_test(img1)
    img2 = preprocess_test(img2)

    similarity = siamese_model.predict([img1,img2])[0][0]

    print("Similarity Score:", similarity)

    if similarity > 0.7:
        print("Prediction: Genuine")
    else:
        print("Prediction: Forged")

In [ ]:
from google.colab import files

uploaded = files.upload()

files_list = list(uploaded.keys())

verify_signature(files_list[0], files_list[1])